# 01 — Split raw data & export CSV

**Mục tiêu**: tách dataset raw thành train/test 90/10 và xuất ra **CSV** theo đúng tên yêu cầu nộp bài.

**Input**:
- `data/raw/data.parquet` — dataset gốc (606,878 dòng × 14 cột)

**Output** (đặt trong `data/raw/`):
- `raw_data_train.csv` — 90% raw, dùng cho clean + train clustering
- `raw_data_test.csv`  — 10% raw, hold-out để kiểm tra pipeline

**Cấu hình**:
- `test_size = 0.1`
- `random_state = 42` (đồng bộ với các notebook khác)
- `stratify = year` — giữ phân phối các năm 2022–2026 đều ở cả train và test

## 1. Setup

In [19]:
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

RAW_PATH          = Path('../data/raw/data.parquet')
OUT_TRAIN_CSV     = Path('../data/raw/raw_data_train.csv')
OUT_TEST_CSV      = Path('../data/raw/raw_data_test.csv')
OUT_TRAIN_PARQUET = Path('../data/raw/raw_data_train.parquet')
OUT_TEST_PARQUET  = Path('../data/raw/raw_data_test.parquet')

RANDOM_STATE = 42
TEST_SIZE    = 0.10
STRATIFY_COL = 'year'

sns.set_style('whitegrid')

FIGSIZE = (12, 5)
TOP_K = 15

## 2. Load raw

In [20]:
df = pd.read_parquet(RAW_PATH)

print(f'Loaded: {len(df):,} rows × {df.shape[1]} columns')
print(f'Memory: {df.memory_usage(deep=True).sum() / 1024**2:.1f} MB')
print()
print('Phân phối year:')
print(df[STRATIFY_COL].value_counts().sort_index().to_string())

Loaded: 606,878 rows × 14 columns
Memory: 1217.4 MB

Phân phối year:
year
2022    111033
2023    129674
2024    159049
2025    160386
2026     46736


## 3. Thông tin tổng quan về tập dữ liệu thô

**Mục tiêu:**

- Kiểm tra shape, kiểu dữ liệu, missing values
- Quan sát nhanh cấu trúc dataset trước khi split
- Phát hiện các cột có khả năng cần clean mạnh ở notebook sau

In [21]:
## 3. Dataset overview

print('=' * 80)
print('DATASET OVERVIEW')
print('=' * 80)

print(f'Số dòng   : {len(df):,}')
print(f'Số cột    : {df.shape[1]}')
print(f'Bộ nhớ RAM: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB')

print('\nDanh sách cột:')
print(df.columns.tolist())

print('\nKiểu dữ liệu:')
display(df.dtypes.to_frame('dtype'))

# =========================================================
# Missing values
# =========================================================

missing_df = pd.DataFrame({
    'missing_count': df.isnull().sum(),
    'missing_pct': (df.isnull().mean() * 100).round(2)
}).sort_values('missing_pct', ascending=False)

print('\nMissing values:')
display(missing_df)

# =========================================================
# Duplicate rows
# =========================================================

duplicate_count = df.duplicated().sum()

print(f'\nDuplicate rows: {duplicate_count:,} '
      f'({duplicate_count / len(df) * 100:.2f}%)')

# =========================================================
# Preview
# =========================================================

print('\nSample dữ liệu:')
display(df.head(5))

DATASET OVERVIEW
Số dòng   : 606,878
Số cột    : 14
Bộ nhớ RAM: 1217.37 MB

Danh sách cột:
['id', 'job_title', 'company_name', 'salary', 'location', 'job_type', 'job_industry', 'experience_level', 'education_level', 'job_position', 'job_description', 'benefits', 'requirements', 'year']

Kiểu dữ liệu:


,dtype
id,int64
job_title,str
company_name,str
salary,str
location,str
job_type,str
job_industry,str
experience_level,str
education_level,str
job_position,str



Missing values:


,missing_count,missing_pct
benefits,3937,0.65
requirements,3736,0.62
job_position,447,0.07
company_name,106,0.02
id,0,0.00
job_title,20,0.00
salary,1,0.00
location,9,0.00
job_type,0,0.00
job_industry,0,0.00



Duplicate rows: 0 (0.00%)

Sample dữ liệu:


,id,job_title,company_name,salary,location,job_type,job_industry,experience_level,education_level,job_position,job_description,benefits,requirements,year
0,1,Java Software Engineer - MSCV 233-2,CÔNG TY TNHH QUICK VIỆT NAM,26 - 36 triệu,Hồ Chí Minh,Toàn thời gian,IT phần mềm,3 năm,Cao đẳng,Nhân Viên,Công ty vốn Nhật chuyên cung cấp dịch vụ điện ...,"Lương: 1,000 ~ 1,400 USD (Gross) Phúc lợi: - B...",Yêu cầu: - Từ 25 ~ 39 tuổi - Trên 3 năm kinh n...,2026
1,2,Software Engineer (Backend/Frontend) - Tiếng T...,CÔNG TY TNHH SONIC FUSION,15 - 35 triệu,Hồ Chí Minh,Toàn thời gian,IT phần mềm,Dưới 1 năm,Không,Nhân Viên,Đọc hiểu tài liệu kỹ thuật và làm việc với đối...,Chế độ bảo hiểm và nghỉ phép đầy đủ theo quy đ...,Thành thạo ít nhất một trong các ngôn ngữ: Pyt...,2026
2,3,"Lead Software Engineer – Desktop (Remote, Engl...",CÔNG TY TNHH PHẦN MỀM GIÁ TRỊ TÍNH TOÁN VÀ ỨNG...,4000 usd,Hà Nội,Toàn thời gian,IT phần mềm,5 năm,Không,Trưởng Nhóm,This role is ideal for a senior developer who ...,Flexible working style Collaborate with divers...,Qualifications Very strong computer science fu...,2026
3,4,Lâp Trình Viên/ Software Engineer/ IT,CÔNG TY TNHH ĐẦU TƯ THƯƠNG MẠI VÀ DỊCH VỤ PHÁT...,10 - 15 triệu,Hà Nội,Toàn thời gian,IT phần mềm,Không,Không,Nhân Viên,"Tham gia vào việc review code, đảm bảo chất lư...",Cơ hội được đào tạo và phát triển bản thân tro...,Có hiểu biết về cấu trúc dữ liệu và giải thuật...,2026
4,5,Software Engineer (Fintech),CÔNG TY TNHH CASSO,14 - 20 triệu,Hồ Chí Minh,Toàn thời gian,IT phần mềm,1 năm,Không,Nhân Viên,Chuẩn hóa dữ liệu: Phối hợp cùng đội ngũ Backe...,Cơ hội lên lead nhanh nếu chứng minh được năng...,Kỹ năng lập trình: Thành thạo ít nhất một ngôn...,2026


## 4. Stratified split 90/10

Stratify theo `year` để train/test có cùng tỉ lệ các năm — quan trọng vì dataset chứa data đa năm và mỗi năm có thể có distribution khác nhau.

In [22]:
train_df, test_df = train_test_split(
    df,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    shuffle=True,
    stratify=df[STRATIFY_COL],
)

print(f'Train: {len(train_df):,} rows  ({len(train_df) / len(df) * 100:.2f}%)')
print(f'Test : {len(test_df):,} rows  ({len(test_df) / len(df) * 100:.2f}%)')
print()

compare = pd.DataFrame({
    'full_pct' : df[STRATIFY_COL].value_counts(normalize=True).sort_index() * 100,
    'train_pct': train_df[STRATIFY_COL].value_counts(normalize=True).sort_index() * 100,
    'test_pct' : test_df[STRATIFY_COL].value_counts(normalize=True).sort_index() * 100,
}).round(2)

print('Phân phối year sau khi split (%):')
print(compare.to_string())

Train: 546,190 rows  (90.00%)
Test : 60,688 rows  (10.00%)

Phân phối year sau khi split (%):
      full_pct  train_pct  test_pct
year                               
2022     18.30      18.30     18.30
2023     21.37      21.37     21.37
2024     26.21      26.21     26.21
2025     26.43      26.43     26.43
2026      7.70       7.70      7.70


## 5. Export CSV

- `encoding='utf-8'` để giữ tiếng Việt có dấu.
- `index=False` để không thêm cột index không cần thiết.
- CSV cho 600k dòng sẽ khá lớn (~1 GB cho train) — chấp nhận được vì format được yêu cầu.

In [23]:
%%time
OUT_TRAIN_CSV.parent.mkdir(parents=True, exist_ok=True)

# Parquet — dùng cho pipeline downstream (nhanh hơn 5-10x)
train_df.to_parquet(OUT_TRAIN_PARQUET, compression='snappy', index=False)
test_df.to_parquet (OUT_TEST_PARQUET,  compression='snappy', index=False)
print(f'✓ Saved train parquet → {OUT_TRAIN_PARQUET}  ({OUT_TRAIN_PARQUET.stat().st_size / 1024**2:.1f} MB)')
print(f'✓ Saved test  parquet → {OUT_TEST_PARQUET}   ({OUT_TEST_PARQUET.stat().st_size / 1024**2:.1f} MB)')

# CSV — artifact để nộp bài (đúng yêu cầu đồ án)
train_df.to_csv(OUT_TRAIN_CSV, index=False, encoding='utf-8')
test_df.to_csv (OUT_TEST_CSV,  index=False, encoding='utf-8')
print(f'✓ Saved train CSV → {OUT_TRAIN_CSV}  ({OUT_TRAIN_CSV.stat().st_size / 1024**2:.1f} MB)')
print(f'✓ Saved test  CSV → {OUT_TEST_CSV}   ({OUT_TEST_CSV.stat().st_size / 1024**2:.1f} MB)')

✓ Saved train parquet → ../data/raw/raw_data_train.parquet  (456.5 MB)
✓ Saved test  parquet → ../data/raw/raw_data_test.parquet   (50.6 MB)
✓ Saved train CSV → ../data/raw/raw_data_train.csv  (1053.5 MB)
✓ Saved test  CSV → ../data/raw/raw_data_test.csv   (117.1 MB)
CPU times: user 19.1 s, sys: 1.21 s, total: 20.3 s
Wall time: 22.1 s


## 6. Sanity check

Đọc lại CSV để xác nhận shape, dtype, encoding tiếng Việt vẫn đúng.

In [24]:
train_reload = pd.read_csv(OUT_TRAIN_CSV, encoding='utf-8')
test_reload  = pd.read_csv(OUT_TEST_CSV,  encoding='utf-8')

print(f'Train reload: {train_reload.shape}  (expected {train_df.shape})')
print(f'Test  reload: {test_reload.shape}   (expected {test_df.shape})')
assert train_reload.shape == train_df.shape
assert test_reload.shape  == test_df.shape

print('\nSample 3 dòng train (kiểm tra tiếng Việt):')
print(train_reload[['job_title', 'location', 'salary']].head(3).to_string(index=False))

Train reload: (546190, 14)  (expected (546190, 14))
Test  reload: (60688, 14)   (expected (60688, 14))

Sample 3 dòng train (kiểm tra tiếng Việt):
                                                job_title                                                                                                                            location                     salary
      [Quảng Bình - Vincom Quảng Bình] Tổ Trưởng Cửa Hàng                           Vincom Quảng Bình | Tầng 26, Tòa nhà CII Tower, số 152 Điện Biên Phủ, Phường Thạnh Mỹ Tây, TP Hồ Chí Minh  7.900.000 - 8.700.000 VND
  Assistant Manager Of Administration And Human Resources        40 đường 40 KDC Vạn Phúc, phường Hiệp Bình Phước | 40 Đường số 40, Phường Hiệp Bình Phước, Thành phố Thủ Đức, TP Hồ Chí Minh                      0 VND
Nhân Viên CSKH (Tư Vấn Chốt Đơn) - Lương Từ 10Tr - Quận 2 Số 12 đường 7A, Phường An Phú | Số 15 đường 54, phường Thảo Điền | Số 6 Đường 31C, Phường An Phú, Thành phố Thủ Đức, TP Hồ Chí Minh 6.000.000 - 

## 7. Tóm tắt

**Đã làm**:
1. Load `data/raw/data.parquet` (606,878 dòng).
2. Phân tích tổng quan tập dữ liệu:
- kiểm tra shape,
- kiểu dữ liệu,
- missing values,
- duplicate rows,
- mức sử dụng bộ nhớ,
- preview dữ liệu mẫu.

3. Split 90/10 stratified theo `year`, `random_state=42`.
4. Export 2 file CSV: `raw_data_train.csv`, `raw_data_test.csv` (UTF-8, không index).
5. Verify bằng cách đọc lại CSV.

**Next**: chạy `02_1_clean_data.ipynb` để clean — notebook đó cần được sửa lại để đọc từ `raw_data_train.csv` + `raw_data_test.csv` và xuất ra `clean_data_train.csv` + `clean_data_test.csv`.